<a href="https://colab.research.google.com/github/RAJANIKANT2907/Data-Science-Projects-and-Assignments/blob/main/LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import files
uploaded = files.upload()


Saving diabetes__1_.csv to diabetes__1_.csv


In [7]:
# =============================================================
#  Logistic Regression Assignment – Diabetes Dataset
# =============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import os
os.makedirs('/home/claude/plots', exist_ok=True)

# ---------------------------------------------------------------
# 1.  DATA EXPLORATION
# ---------------------------------------------------------------
print("=" * 60)
print("1. DATA EXPLORATION")
print("=" * 60)

df = pd.read_csv('diabetes__1_.csv')

print("\n--- Shape ---")
print(df.shape)

print("\n--- First 5 rows ---")
print(df.head())

print("\n--- Feature Types ---")
print(df.dtypes)

print("\n--- Summary Statistics ---")
print(df.describe().round(2))

print("\n--- Target Distribution ---")
print(df['Outcome'].value_counts())
print(f"  Class balance: {df['Outcome'].mean():.2%} positive (diabetic)")

# --- Visualisation 1: Histograms ---
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
for i, col in enumerate(df.columns):
    ax = axes[i // 3][i % 3]
    color = '#e74c3c' if col == 'Outcome' else '#3498db'
    df[col].hist(ax=ax, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig('/home/claude/plots/01_histograms.png', dpi=150)
plt.close()
print("\n[Plot saved] 01_histograms.png")

# --- Visualisation 2: Box plots by Outcome ---
features = [c for c in df.columns if c != 'Outcome']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Feature Distributions by Outcome', fontsize=15, fontweight='bold')
for i, col in enumerate(features):
    ax = axes[i // 4][i % 4]
    df.boxplot(column=col, by='Outcome', ax=ax,
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Outcome (0=No Diabetes, 1=Diabetes)')
    plt.sca(ax); plt.title(col)
plt.suptitle('Box Plots by Outcome', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/plots/02_boxplots.png', dpi=150)
plt.close()
print("[Plot saved] 02_boxplots.png")

# --- Visualisation 3: Correlation heatmap ---
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/home/claude/plots/03_correlation_heatmap.png', dpi=150)
plt.close()
print("[Plot saved] 03_correlation_heatmap.png")

print("\n--- Key Correlations with Outcome ---")
print(corr['Outcome'].sort_values(ascending=False).to_string())

# ---------------------------------------------------------------
# 2.  DATA PREPROCESSING
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("2. DATA PREPROCESSING")
print("=" * 60)

# Columns where 0 is physiologically impossible → treat as missing
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("\n--- Zero counts (before imputation) ---")
for col in zero_cols:
    n = (df[col] == 0).sum()
    print(f"  {col}: {n} zeros ({n/len(df):.1%})")

# Replace 0s with NaN, then impute with median
df_clean = df.copy()
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)
medians = df_clean[zero_cols].median()
df_clean[zero_cols] = df_clean[zero_cols].fillna(medians)

print("\n--- Missing values after imputation ---")
print(df_clean.isnull().sum())

# No categorical variables in this dataset – all numeric
print("\n--- Categorical encoding: Not required (all features are numeric) ---")

# Train / Test split
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\n--- Train/Test Split ---")
print(f"  Training samples : {len(X_train)}")
print(f"  Testing  samples : {len(X_test)}")

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ---------------------------------------------------------------
# 3.  MODEL BUILDING
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("3. MODEL BUILDING")
print("=" * 60)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)
print("\nLogistic Regression model trained successfully.")

# ---------------------------------------------------------------
# 4.  MODEL EVALUATION
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("4. MODEL EVALUATION")
print("=" * 60)

y_pred       = model.predict(X_test_sc)
y_pred_proba = model.predict_proba(X_test_sc)[:, 1]

acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_proba)

print(f"\n  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")

print("\n--- Full Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, scaler.transform(X), y, cv=cv, scoring='roc_auc')
print(f"\n--- 5-Fold Cross-Validation ROC-AUC ---")
print(f"  Scores : {cv_scores.round(4)}")
print(f"  Mean   : {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}")

# --- Visualisation 4: Confusion Matrix ---
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/plots/04_confusion_matrix.png', dpi=150)
plt.close()
print("\n[Plot saved] 04_confusion_matrix.png")

# --- Visualisation 5: ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2980b9', lw=2.5,
         label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='#95a5a6', linestyle='--', lw=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.15, color='#2980b9')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve – Logistic Regression', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/home/claude/plots/05_roc_curve.png', dpi=150)
plt.close()
print("[Plot saved] 05_roc_curve.png")

# ---------------------------------------------------------------
# 5.  INTERPRETATION
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("5. INTERPRETATION")
print("=" * 60)

coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': model.coef_[0],
    'Odds Ratio' : np.exp(model.coef_[0])
}).sort_values('Coefficient', ascending=False)

print("\n--- Logistic Regression Coefficients ---")
print(coef_df.to_string(index=False))

# --- Visualisation 6: Coefficient Plot ---
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient Value (Standardised)', fontsize=12)
ax.set_title('Feature Importance – Logistic Regression Coefficients\n'
             '(Red = increases diabetes risk | Green = decreases risk)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('/home/claude/plots/06_coefficients.png', dpi=150)
plt.close()
print("[Plot saved] 06_coefficients.png")

print("\n--- Feature Significance Discussion ---")
for _, row in coef_df.iterrows():
    direction = "INCREASES" if row['Coefficient'] > 0 else "DECREASES"
    print(f"  {row['Feature']:<30} coef={row['Coefficient']:+.3f}  "
          f"OR={row['Odds Ratio']:.3f}  → {direction} diabetes risk")

# ---------------------------------------------------------------
# INTERVIEW ANSWERS
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("INTERVIEW QUESTIONS")
print("=" * 60)

print("""
Q1. What is the difference between Precision and Recall?
  • Precision = TP / (TP + FP)
    Out of all samples predicted POSITIVE, how many are actually positive.
    High precision → few false alarms.

  • Recall    = TP / (TP + FN)
    Out of all actual POSITIVES, how many did we correctly identify.
    High recall → few missed positives.

  Trade-off: Improving one often hurts the other. In medical diagnosis
  (e.g. diabetes) recall is usually more critical – missing a sick patient
  (false negative) is costlier than a false alarm.

Q2. What is Cross-Validation and why is it important?
  Cross-validation (CV) splits the dataset into k folds, trains the model
  on k-1 folds, and validates on the remaining fold – repeating k times.

  Importance in binary classification:
  • Provides a robust, low-variance estimate of generalisation performance.
  • Detects overfitting that a single train/test split can miss.
  • With StratifiedKFold, each fold preserves class proportions, which is
    critical for imbalanced datasets like this one (~35% positive).
  • Helps in hyperparameter tuning without touching the held-out test set.
""")

print("=" * 60)
print("All done! Plots saved to /home/claude/plots/")
print("=" * 60)

1. DATA EXPLORATION

--- Shape ---
(768, 9)

--- First 5 rows ---
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

--- Feature Types ---
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float6